In [ ]:
#Install Kaggle API
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()

{}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d risangbaskoro/wlasl-processed
!unzip -q wlasl-processed.zip

Dataset URL: https://www.kaggle.com/datasets/risangbaskoro/wlasl-processed
License(s): other
wlasl-processed.zip: Skipping, found more recently modified local copy (use --force to force download)
replace WLASL_v0.3.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
#Load JSON Labels
import json

with open("/content/WLASL_v0.3.json") as f:
    data_json = json.load(f)

print("Total words:", len(data_json))

Total words: 2000


In [ ]:
#Extract Frames
import cv2
import numpy as np
from tqdm import tqdm

IMG_SIZE = 64
MAX_FRAMES = 10   # take 10 frames per video

X = []
y = []

for item in tqdm(data_json[:200]):   # limit for speed

    label = item['gloss']

    for instance in item['instances']:
        video_id = instance['video_id']
        video_path = f"/content/videos/{video_id}.mp4"

        if not os.path.exists(video_path):
            continue

        cap = cv2.VideoCapture(video_path)

        frames = []
        count = 0

        while count < MAX_FRAMES:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            frame = frame / 255.0

            frames.append(frame)
            count += 1

        cap.release()

        if len(frames) == MAX_FRAMES:
            X.append(frames)
            y.append(label)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

  0%|          | 0/200 [00:00<?, ?it/s]


NameError: name 'os' is not defined

In [ ]:
#Encode Labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

NameError: name 'y' is not defined

In [ ]:
#Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

NameError: name 'X' is not defined

In [ ]:
#Build Model (CNN)
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv3D(32, (3,3,3), activation='relu',
                  input_shape=(10,64,64,3)),
    layers.MaxPooling3D((2,2,2)),

    layers.Conv3D(64, (3,3,3), activation='relu'),
    layers.MaxPooling3D((2,2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(set(y)), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


NameError: name 'y' is not defined

In [ ]:
#Train Model
history = model.fit(
    X_train, y_train,
    epochs=18,
    validation_data=(X_test, y_test)
)

NameError: name 'X_train' is not defined

In [ ]:
model.evaluate(X_test, y_test)

12/12 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - accuracy: 0.0240 - loss: 5.7028


[5.702807903289795, 0.024000000208616257]

In [ ]:
model.save("asl_model.h5")

In [ ]:
pip install opencv-python tensorflow numpy

In [ ]:
# =========================
# 1. Install
# =========================
#!pip install opencv-python matplotlib

# =========================
# 2. Import libraries
# =========================
import cv2
import numpy as np
import tensorflow as tf
import pickle
from google.colab import files

# =========================
# 3. Load trained model
# =========================
model = tf.keras.models.load_model("asl_model.h5")

IMG_SIZE = 64
MAX_FRAMES = 10   # must match training

# =========================
# 4. Load labels
# =========================
try:
    index_to_label = pickle.load(open("labels.pkl", "rb"))
    print("✅ Labels loaded")
except:
    index_to_label = {0: "hello", 1: "thanks", 2: "yes", 3: "no"}  # fallback
    print("⚠️ Using manual labels")

# =========================
# 5. Upload VIDEO
# =========================
uploaded = files.upload()

video_path = list(uploaded.keys())[0]

# =========================
# 6. Read video + extract frames
# =========================
cap = cv2.VideoCapture(video_path)

frames = []

while len(frames) < MAX_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    frame = frame.astype("float32") / 255.0

    frames.append(frame)

cap.release()

# =========================
# 7. Check frames
# =========================
if len(frames) < MAX_FRAMES:
    raise Exception("❌ Not enough frames in video")

# Convert to numpy
input_data = np.array(frames)
input_data = np.expand_dims(input_data, axis=0)

print("Input shape:", input_data.shape)
# expected: (1, 10, 64, 64, 3)

# =========================
# 8. Predict
# =========================
pred = model.predict(input_data, verbose=0)

class_id = np.argmax(pred)
confidence = float(np.max(pred))

# =========================
# 9. Get label
# =========================
label = index_to_label[class_id]

# =========================
# 10. Show result
# =========================
print("🎥 Prediction:", label)
print("📊 Confidence:", round(confidence, 2))

✅ Labels loaded


Saving stock-footage-female-saying-thank-you-in-sign-language-teacher-showing-words-in-asl-tutorial.mp4 to stock-footage-female-saying-thank-you-in-sign-language-teacher-showing-words-in-asl-tutorial.mp4
Input shape: (1, 10, 64, 64, 3)
🎥 Prediction: bee
📊 Confidence: 0.01


In [ ]:
print(pred)

[[0.00393965 0.00717903 0.00366751 0.00495082 0.00450501 0.00497943
  0.0055456  0.00473056 0.00554827 0.00397692 0.00505658 0.00514507
  0.00565518 0.00575841 0.0054708  0.00342039 0.00626584 0.00643347
  0.00731012 0.00653405 0.00398452 0.00333822 0.00548616 0.00508428
  0.00520111 0.0054422  0.00675186 0.00528313 0.00414887 0.00425051
  0.005049   0.00452643 0.00670505 0.00357219 0.00374257 0.00564845
  0.00390098 0.00652805 0.00513924 0.00481877 0.00345194 0.00880594
  0.00347699 0.00438266 0.00472182 0.00490239 0.00493414 0.00497192
  0.00398001 0.00505269 0.00154613 0.00402239 0.00630403 0.00385298
  0.00390632 0.00759406 0.00478781 0.00315278 0.00588718 0.00369721
  0.00450806 0.00372077 0.00508669 0.00615571 0.00443604 0.00662805
  0.00342626 0.00427564 0.00490305 0.0061712  0.00603763 0.00538725
  0.00383812 0.00570821 0.00408721 0.00587827 0.00495355 0.00575174
  0.00337087 0.00333289 0.0079788  0.00505712 0.00888367 0.00544255
  0.00413834 0.00554577 0.004846   0.00339614 0.